In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv("/content/all_stocks_5yr.csv")

# EDA

In [15]:
# 1. ดู 5 แถวแรกของข้อมูล
display(df.head())

# 2. ดูโครงสร้างข้อมูล และประเภทของคอลัมน์ (เพื่อตรวจสอบว่า date เป็น datetime หรือยัง)
display(df.info())

# 3. ดูค่าสถิติพื้นฐาน (Mean, Max, Min, Std)
display(df.describe())

# 4. ตรวจสอบว่ามีข้อมูลสูญหาย (Missing Values) หรือไม่
display(df.isnull().sum())

,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 619040 entries, 0 to 619039
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    619040 non-null  object 
 1   open    619029 non-null  float64
 2   high    619032 non-null  float64
 3   low     619032 non-null  float64
 4   close   619040 non-null  float64
 5   volume  619040 non-null  int64  
 6   Name    619040 non-null  object 
dtypes: float64(4), int64(1), object(2)
memory usage: 33.1+ MB


None

,open,high,low,close,volume
count,619029.000000,619032.000000,619032.000000,619040.000000,6.190400e+05
mean,83.023334,83.778311,82.256096,83.043763,4.321823e+06
std,97.378769,98.207519,96.507421,97.389748,8.693610e+06
min,1.620000,1.690000,1.500000,1.590000,0.000000e+00
25%,40.220000,40.620000,39.830000,40.245000,1.070320e+06
50%,62.590000,63.150000,62.020000,62.620000,2.082094e+06
75%,94.370000,95.180000,93.540000,94.410000,4.284509e+06
max,2044.000000,2067.990000,2035.110000,2049.000000,6.182376e+08


,0
date,0
open,11
high,8
low,8
close,0
volume,0
Name,0


In [11]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical Variables:")
for col in numerical_cols:
    print(f"- {col}")

print("\nCategorical Variables:")
for col in categorical_cols:
    print(f"- {col}")

Numerical Variables:
- open
- high
- low
- close
- volume

Categorical Variables:
- date
- Name


In [12]:
print("Unique values in categorical columns:")
print("=" * 40)
for col in categorical_cols:
    num_unique = df[col].nunique()
    if num_unique <= 12:
        unique_vals = df[col].unique().tolist()
        print(f"{col}: {num_unique} ค่า ได้แก่ {unique_vals}")
    else:
        print(f"{col}: {num_unique} ค่า")


Unique values in categorical columns:
date: 1259 ค่า
Name: 505 ค่า


In [16]:
duplicate_rows = df[df.duplicated()]
print(duplicate_rows)

Empty DataFrame
Columns: [date, open, high, low, close, volume, Name]
Index: []


In [17]:
print("\n--- 🚨 ตรวจสอบค่าติดลบใน Numerical Variables ---")
has_negative = False

for col in numerical_cols:
    negative_rows = df[df[col] < 0]

    if not negative_rows.empty:
        print(f"⚠️ พบค่าติดลบในคอลัมน์ '{col}':")
        print(negative_rows[['date', col]])
        has_negative = True

if not has_negative:
    print("✅ ข้อมูลปกติ: ไม่พบค่าติดลบในคอลัมน์ตัวเลขเลยครับ")


--- 🚨 ตรวจสอบค่าติดลบใน Numerical Variables ---
✅ ข้อมูลปกติ: ไม่พบค่าติดลบในคอลัมน์ตัวเลขเลยครับ


In [22]:
# สมมติว่า df ของคุณมีข้อมูลหุ้นหลายตัวแล้ว
# df = pd.read_csv('all_stocks_5yr.csv')

numerical_cols = ['open', 'high', 'low', 'close', 'volume']

# 1. สร้างฟังก์ชันเช็ก Outlier ด้วย IQR (รับค่าเป็น Series ของหุ้นแต่ละตัว)
def check_outlier(x):
    Q1 = x.quantile(0.25)
    Q3 = x.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # คืนค่าเป็น True ถ้าเป็น Outlier, False ถ้าปกติ
    return (x < lower_bound) | (x > upper_bound)

# 2. ใช้ groupby('Name') เพื่อให้ฟังก์ชันคำนวณแยกตามหุ้น
# .transform() จะช่วยคำนวณและแปะค่ากลับไปที่แถวเดิมของ DataFrame
outlier_masks = df.groupby('Name')[numerical_cols].transform(check_outlier)

# เปลี่ยนชื่อคอลัมน์ให้สื่อความหมาย (เช่น 'close' -> 'outlier_close')
outlier_masks.columns = [f'outlier_{col}' for col in numerical_cols]

# เอาผลลัพธ์ไปแปะรวมกับ df เดิมเพื่อดูง่ายๆ
df_with_outliers = pd.concat([df, outlier_masks], axis=1)

# 3. ลองกรองดูว่าตอนนี้มี Outlier จริงๆ (ที่ผิดปกติสำหรับหุ้นตัวนั้นๆ) กี่แถว
real_outliers_close = df_with_outliers[df_with_outliers['outlier_close'] == True]

print(f"เจอ Outlier ของราคาปิด (ที่เทียบกับตัวเองแล้ว) จำนวน: {len(real_outliers_close)} แถว")

เจอ Outlier ของราคาปิด (ที่เทียบกับตัวเองแล้ว) จำนวน: 9934 แถว


In [23]:
# นับจำนวนบริษัทหุ้น (Name) ที่ไม่ซ้ำกัน ในกลุ่มที่เกิด Outlier
outlier_stock_count = real_outliers_close['Name'].nunique()
print(f"จาก 9,934 แถว พบว่าเป็นของหุ้นจำนวน: {outlier_stock_count} บริษัท")

# ดูสัดส่วน % ของ Outlier เทียบกับข้อมูลทั้งหมด
total_rows = len(df)
percent_outliers = (len(real_outliers_close) / total_rows) * 100
print(f"คิดเป็น {percent_outliers:.2f}% ของข้อมูลทั้งหมด {total_rows} แถว")

จาก 9,934 แถว พบว่าเป็นของหุ้นจำนวน: 184 บริษัท
คิดเป็น 1.60% ของข้อมูลทั้งหมด 619040 แถว


# Cleaning Data

In [25]:
# 1. เปลี่ยนคอลัมน์ date เป็นประเภท datetime
df['date'] = pd.to_datetime(df['date'])

# 2. เปลี่ยนคอลัมน์ Name เป็นประเภท string (ใน Pandas เวอร์ชันใหม่ๆ รองรับ 'string' แล้วครับ)
df['Name'] = df['Name'].astype('string')

# ตรวจสอบความถูกต้องหลังเปลี่ยนประเภทข้อมูล
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 619040 entries, 0 to 619039
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    619040 non-null  datetime64[ns]
 1   open    619029 non-null  float64       
 2   high    619032 non-null  float64       
 3   low     619032 non-null  float64       
 4   close   619040 non-null  float64       
 5   volume  619040 non-null  int64         
 6   Name    619040 non-null  string        
dtypes: datetime64[ns](1), float64(4), int64(1), string(1)
memory usage: 33.1 MB
None


In [26]:
# สมมติว่าข้อมูลของคุณอยู่ในตัวแปร df
print("--- 🔍 ก่อนลบ Missing Values ---")
print(df.isnull().sum())

# ทำการลบแถวที่มี Missing Value ทิ้งทั้งหมด (Listwise Deletion)
df_clean = df.dropna()

print("\n--- ✅ หลังลบ Missing Values ---")
print(df_clean.isnull().sum())

# เช็กจำนวนแถวที่เหลืออยู่
print(f"\nเหลือข้อมูลทั้งหมด: {len(df_clean)} แถว")

--- 🔍 ก่อนลบ Missing Values ---
date       0
open      11
high       8
low        8
close      0
volume     0
Name       0
dtype: int64

--- ✅ หลังลบ Missing Values ---
date      0
open      0
high      0
low       0
close     0
volume    0
Name      0
dtype: int64

เหลือข้อมูลทั้งหมด: 619029 แถว
